# Training Table Walkthrough

This notebook explains and audits the pass-detection training table created from the Step 2 master join table.

Expected contract:

- the master join is split by match before feature engineering
- each output row is one non-overlapping 5-frame window, about 0.5 seconds
- train, validation, and test are written as separate Parquet files
- the target is `is_pass`, derived from `primary_event == "PASS"`
- windows where all ball position values are missing are skipped

Plain-language companion doc: `docs/training_table_walkthrough.md`.

## 1. Setup

This cell makes the notebook work whether it is opened from the project root or from the `notebooks/` folder.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from driblab.config import CONFIG_PATH, MODEL_BASE_DATA_DIR
from driblab.features import match_splits
from driblab.features.training_table import OUTPUT_COLUMNS

MASTER_JOIN_PATH = MODEL_BASE_DATA_DIR / "master_join_table.parquet"
TRAINING_PATHS = {
    "train": MODEL_BASE_DATA_DIR / "training_table_train.parquet",
    "validation": MODEL_BASE_DATA_DIR / "training_table_validation.parquet",
    "test": MODEL_BASE_DATA_DIR / "training_table_test.parquet",
}
SUMMARY_PATHS = {
    split: MODEL_BASE_DATA_DIR / f"training_table_summary_{split}.csv"
    for split in TRAINING_PATHS
}

print(PROJECT_ROOT)

/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab


## 2. Rebuild the training tables

Run this from a terminal at the project root when you want to regenerate the files:

```bash
conda activate driblabvenv
PYTHONPATH=src python -m driblab.features.training_table
```

The command reads `master_join_table.parquet` and `config.yaml`, then writes one table and one summary CSV per split.

In [2]:
assert MASTER_JOIN_PATH.exists(), f"Missing {MASTER_JOIN_PATH}. Run Step 2 first."
for split, path in TRAINING_PATHS.items():
    assert path.exists(), f"Missing {path}. Run: PYTHONPATH=src python -m driblab.features.training_table"
for split, path in SUMMARY_PATHS.items():
    assert path.exists(), f"Missing {path}. Run the training table builder."

file_inventory = pd.DataFrame(
    [
        {
            "split": split,
            "table_path": str(TRAINING_PATHS[split].relative_to(PROJECT_ROOT)),
            "table_mb": TRAINING_PATHS[split].stat().st_size / 1024 / 1024,
            "summary_path": str(SUMMARY_PATHS[split].relative_to(PROJECT_ROOT)),
        }
        for split in TRAINING_PATHS
    ]
)
display(file_inventory)

,split,table_path,table_mb,summary_path
0,train,data/processed/model_base/training_table_train...,6.641921,data/processed/model_base/training_table_summa...
1,validation,data/processed/model_base/training_table_valid...,1.763017,data/processed/model_base/training_table_summa...
2,test,data/processed/model_base/training_table_test....,1.660271,data/processed/model_base/training_table_summa...


## 3. Load summaries

The summary CSVs are the quickest health check. They count windows, pass windows, no-event windows, match coverage, and period coverage.

In [3]:
summaries = pd.concat(
    [pd.read_csv(path) for path in SUMMARY_PATHS.values()],
    ignore_index=True,
)
summaries["pass_percentage"] = summaries["pass_percentage"].round(2)
display(summaries)

,split_name,total_windows,pass_windows,no_event_windows,pass_percentage,unique_matches,unique_periods
0,train,161505,15554,140892,9.63,23,2
1,validation,36719,3567,32044,9.71,5,2
2,test,35085,3725,30251,10.62,5,2


## 4. Check schemas

All split tables should have the same 16 columns, in the same order.

In [4]:
schema_checks = []
for split, path in TRAINING_PATHS.items():
    columns = pq.ParquetFile(path).schema_arrow.names
    schema_checks.append(
        {
            "split": split,
            "column_count": len(columns),
            "matches_expected_order": columns == OUTPUT_COLUMNS,
        }
    )

display(pd.DataFrame(schema_checks))
display(pd.DataFrame({"column_order": OUTPUT_COLUMNS}))

,split,column_count,matches_expected_order
0,train,16,True
1,validation,16,True
2,test,16,True


,column_order
0,t.match_id
1,t.period
2,window_time
3,data_split
4,primary_event
5,is_pass
6,secondary_events
7,ball_x_avg
8,ball_y_avg
9,ball_z_avg


## 5. Preview rows

A few rows from each split show the modelling grain: match, period, window time, primary event, target, ball features, closest-player features, and the player-change signal.

In [5]:
preview_columns = [
    "t.match_id",
    "t.period",
    "window_time",
    "data_split",
    "primary_event",
    "is_pass",
    "ball_x_avg",
    "ball_y_avg",
    "ball_z_avg",
    "ball_speed_avg",
    "closest_player_team_start",
    "closest_player_team_end",
    "player_changed_same_team",
]

for split, path in TRAINING_PATHS.items():
    print(split)
    df_preview = pd.read_parquet(path, columns=preview_columns).head(5)
    display(df_preview)

train


,t.match_id,t.period,window_time,data_split,primary_event,is_pass,ball_x_avg,ball_y_avg,ball_z_avg,ball_speed_avg,closest_player_team_start,closest_player_team_end,player_changed_same_team
0,678949,1,3.0,train,no event,0,41.6125,42.5025,0.0165,1.278106,unknown,1925,0
1,678949,1,3.5,train,no event,0,37.0460,44.2500,0.1382,1.276188,1925,1925,0
2,678949,1,4.0,train,no event,0,32.2460,46.1860,0.1810,1.196294,1925,1925,0
3,678949,1,4.5,train,no event,0,27.1880,42.1320,0.0308,0.759206,1925,1925,0
4,678949,1,5.0,train,PASS,1,24.5520,41.7680,0.0000,0.493233,1925,1925,0


validation


,t.match_id,t.period,window_time,data_split,primary_event,is_pass,ball_x_avg,ball_y_avg,ball_z_avg,ball_speed_avg,closest_player_team_start,closest_player_team_end,player_changed_same_team
0,689526,1,0.5,validation,PASS,1,51.0850,4.9150,4.91650,7.521740,unknown,unknown,0
1,689526,1,1.0,validation,no event,0,38.8225,17.5525,17.55125,3.192972,unknown,7194,0
2,689526,1,1.5,validation,no event,0,36.9500,34.0640,34.06300,32.151111,7194,7194,0
3,689526,1,2.0,validation,no event,0,37.6920,42.7940,42.79360,25.914413,7194,7186,0
4,689526,1,2.5,validation,no event,0,50.8850,61.3850,61.38500,14.916471,unknown,unknown,0


test


,t.match_id,t.period,window_time,data_split,primary_event,is_pass,ball_x_avg,ball_y_avg,ball_z_avg,ball_speed_avg,closest_player_team_start,closest_player_team_end,player_changed_same_team
0,713910,1,0.5,test,no event,0,63.436667,39.493333,0.8630,38.783023,unknown,unknown,0
1,713910,1,1.0,test,no event,0,59.024000,70.822000,0.6836,10.168189,7188,7188,1
2,713910,1,1.5,test,no event,0,58.142000,78.540000,0.6050,0.934027,7188,7188,0
3,713910,1,2.0,test,PASS,1,68.982000,8.804000,0.6102,9.726110,7188,7188,1
4,713910,1,2.5,test,no event,0,69.990000,1.000000,0.7900,NaN,unknown,7188,0


## 6. Confirm split isolation

The training table should contain only matches assigned to that split in `config.yaml`. This is the key leakage-prevention check.

In [6]:
splits = match_splits.load_match_splits(CONFIG_PATH)
split_checks = []

for split, path in TRAINING_PATHS.items():
    df = pd.read_parquet(path, columns=["t.match_id", "data_split"])
    expected_matches = set(map(str, splits[split]))
    actual_matches = set(df["t.match_id"].astype(str).unique())
    split_checks.append(
        {
            "split": split,
            "all_rows_have_expected_label": bool((df["data_split"] == split).all()),
            "actual_matches": len(actual_matches),
            "expected_matches": len(expected_matches),
            "unexpected_matches": sorted(actual_matches - expected_matches),
            "missing_matches": sorted(expected_matches - actual_matches),
        }
    )

display(pd.DataFrame(split_checks))

,split,all_rows_have_expected_label,actual_matches,expected_matches,unexpected_matches,missing_matches
0,train,True,23,23,[],[]
1,validation,True,5,5,[],[]
2,test,True,5,5,[],[]


## 7. Target balance by split

The pass rate should be checked before modelling. Here it is similar across train, validation, and test, which is helpful for evaluation.

In [7]:
target_rows = []
for split, path in TRAINING_PATHS.items():
    df = pd.read_parquet(path, columns=["primary_event", "is_pass"])
    target_rows.append(
        {
            "split": split,
            "windows": len(df),
            "pass_windows": int(df["is_pass"].sum()),
            "pass_percentage": round(df["is_pass"].mean() * 100, 2),
            "no_event_windows": int((df["primary_event"] == "no event").sum()),
        }
    )

display(pd.DataFrame(target_rows))

,split,windows,pass_windows,pass_percentage,no_event_windows
0,train,161505,15554,9.63,140892
1,validation,36719,3567,9.71,32044
2,test,35085,3725,10.62,30251


## 8. Event distribution

`PASS` becomes the positive class. Other selected events and `no event` become negative examples.

In [8]:
event_distribution = []
for split, path in TRAINING_PATHS.items():
    events = pd.read_parquet(path, columns=["primary_event"])
    counts = events["primary_event"].value_counts().rename_axis("primary_event")
    split_counts = counts.reset_index(name="windows")
    split_counts.insert(0, "split", split)
    event_distribution.append(split_counts)

event_distribution = pd.concat(event_distribution, ignore_index=True)
display(event_distribution)

,split,primary_event,windows
0,train,no event,140892
1,train,PASS,15554
2,train,BALL TOUCH,2238
3,train,TACKLE,741
4,train,BALL RECOVERY,646
5,train,AERIAL,605
6,train,TAKEON,501
7,train,FOUL,328
8,validation,no event,32044
9,validation,PASS,3567


## 9. Audit the confirmed skip rule

The builder skips a 5-frame window only when all `t.ball_x`, `t.ball_y`, and `t.ball_z` values are missing across the whole window.

This cell estimates how many complete 5-frame windows existed in the master join after split assignment, then compares that with the generated output row counts.

In [9]:
master_cols = [
    "t.match_id",
    "t.period",
    "t.frame",
    "t.ball_x",
    "t.ball_y",
    "t.ball_z",
]
master = pd.read_parquet(MASTER_JOIN_PATH, columns=master_cols)
master = match_splits.add_data_split_column(
    master,
    splits,
    match_col="t.match_id",
)
master = master.sort_values(["t.match_id", "t.period", "t.frame"])

skip_audit_rows = []
for split, split_df in master.groupby("data_split"):
    if split not in TRAINING_PATHS:
        continue
    complete_windows = 0
    skipped_all_ball_missing = 0
    for _, period_df in split_df.groupby(["t.match_id", "t.period"], sort=False):
        complete_rows = (len(period_df) // 5) * 5
        if complete_rows == 0:
            continue
        ball = period_df.iloc[:complete_rows][["t.ball_x", "t.ball_y", "t.ball_z"]]
        ball = ball.apply(pd.to_numeric, errors="coerce").to_numpy().reshape(-1, 5, 3)
        complete_windows += len(ball)
        skipped_all_ball_missing += int(pd.isna(ball).all(axis=(1, 2)).sum())

    output_windows = int(summaries.loc[summaries["split_name"] == split, "total_windows"].iloc[0])
    skip_audit_rows.append(
        {
            "split": split,
            "complete_5_frame_windows": complete_windows,
            "skipped_all_ball_missing": skipped_all_ball_missing,
            "output_windows": output_windows,
            "complete_minus_skipped": complete_windows - skipped_all_ball_missing,
            "matches_output_count": complete_windows - skipped_all_ball_missing == output_windows,
        }
    )

display(pd.DataFrame(skip_audit_rows))

,split,complete_5_frame_windows,skipped_all_ball_missing,output_windows,complete_minus_skipped,matches_output_count
0,test,58488,23403,35085,35085,True
1,train,279437,117932,161505,161505,True
2,validation,59372,22653,36719,36719,True


## 10. Feature notes

- `ball_x_avg`, `ball_y_avg`, and `ball_z_avg` are raw tracking-coordinate averages.
- `ball_speed_avg` is average raw Euclidean frame-to-frame ball movement.
- `closest_player_*_start` is computed on frame 1 of the window.
- `closest_player_*_end` is computed on frame 5 of the window.
- `player_changed_same_team` is a model feature, not the label.
- `primary_event` is selected by smallest `nearest_timestamp_distance_sec` inside the window.
- `secondary_events` stores the other event types from the same window.
- Missing numeric feature values are kept unless the whole window has no ball position at all.

## 11. Next modelling step

The generated tables are ready to feed into a binary pass classifier. A first baseline can use the numeric columns plus encoded categorical columns for closest-player teams and primary/secondary event context, while keeping the split files separate for evaluation.